In [0]:
%sql
select * from pyspark_scenarios_catalog.source.products

In [0]:
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

# df = spark.read.table('pyspark_scenarios_catalog.source.products')
df = spark.sql("select * from pyspark_scenarios_catalog.source.products")

#DEDUP
df = df.withColumn('dedup_rank', row_number().over(Window.partitionBy('id').orderBy(col('updatedDate').desc()))).filter(col('dedup_rank')==1).drop('dedup_rank')

display(df)

In [0]:
df.write.format('delta').mode('overwrite').save('/Volumes/pyspark_scenarios_catalog/source/db_volume/products_sink/')

In [0]:
"""
If source_df has duplicate ids, the merge will fail because Delta cannot deterministically know which record to keep.
In DLT with Auto-CDC, you can do SEQUENCE BY some_column to automatically pick the latest record per key.
PySpark does not have a direct SEQUENCE BY equivalent in MERGE
You need to DEdup the src df before running merge
* .whenMatchedUpdateAll(condition="src.updatedDate >= trg.updatedDate") - mimics "sequence by" in DLT
   - if source data is refershed, older data will not be overwritten
"""
from delta.tables import DeltaTable

# if dbutils.fs.ls('/Volumes/pyspark_scenarios_catalog/source/db_volume/products_sink/'):
if DeltaTable.isDeltaTable(spark, "/Volumes/pyspark_scenarios_catalog/source/db_volume/products_sink"):
    print('TABLE EXISTS')
    delta_obj = DeltaTable.forPath(spark, '/Volumes/pyspark_scenarios_catalog/source/db_volume/products_sink/')
    delta_obj.alias('trg').merge(
        df.alias('src'),
        "src.id=trg.id")\
        .whenMatchedUpdateAll(condition="src.updatedDate >= trg.updatedDate")\
        .whenNotMatchedInsertAll()\
        .execute()


In [0]:
%sql
select * from delta.`/Volumes/pyspark_scenarios_catalog/source/db_volume/products_sink`

In [0]:
%sql
select * from delta.`/Volumes/pyspark_scenarios_catalog/source/db_volume/products_sink`